# LMP Example 6 - Three-Bus, Congestion on Branch 2 (c1=50, c3=20)
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

> Three-bus system where the expensive generator (G1, \$50/MWh) is at bus 1 and the cheap one (G3, \$20/MWh) is at bus 3. Branch 2 (bus 1→3, 26 MW limit) is binding. Tip: change `Load2` by ±1 MW to confirm LMP[2].


In [ ]:
from pyomo.environ import *
import time

# ─────────────────────────────────────────────────────────────────
# System Data
# ─────────────────────────────────────────────────────────────────
BaseMW = 100
refBus = 3

# Generators: {g: {bus, Pgmin MW, Pgmax MW, cost $/MWh}}
gen_data = {1: {'bus': 1, 'Pgmin': 20, 'Pgmax': 80, 'c': 50}, 2: {'bus': 3, 'Pgmin': 40, 'Pgmax': 90, 'c': 20}}

# Loads: {bus: MW}
load_data = {2: 100}

# Branches: {k: {from, to, x pu, rate MW (None = unlimited)}}
branch_data = {1: {'from': 1, 'to': 2, 'x': 0.1, 'rate': 100}, 2: {'from': 1, 'to': 3, 'x': 0.1, 'rate': 26}, 3: {'from': 2, 'to': 3, 'x': 0.2, 'rate': 100}}


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Build Pyomo Model
# ─────────────────────────────────────────────────────────────────
model = ConcreteModel()

model.BUS    = Set(initialize=sorted(set(gen_data[g]['bus'] for g in gen_data) | set(load_data)))
model.GEN    = Set(initialize=gen_data.keys())
model.BRANCH = Set(initialize=branch_data.keys())

model.busLoad       = Param(model.BUS,    initialize=load_data, default=0)
model.gen_bus       = Param(model.GEN,    initialize={g: gen_data[g]['bus']   for g in gen_data}, within=model.BUS)
model.gen_min       = Param(model.GEN,    initialize={g: gen_data[g]['Pgmin'] for g in gen_data})
model.gen_max       = Param(model.GEN,    initialize={g: gen_data[g]['Pgmax'] for g in gen_data})
model.gen_OpCost    = Param(model.GEN,    initialize={g: gen_data[g]['c']     for g in gen_data})
model.branch_frmbus = Param(model.BRANCH, initialize={k: branch_data[k]['from'] for k in branch_data}, within=model.BUS)
model.branch_tobus  = Param(model.BRANCH, initialize={k: branch_data[k]['to']   for k in branch_data}, within=model.BUS)
model.branch_x      = Param(model.BRANCH, initialize={k: branch_data[k]['x']    for k in branch_data})
model.branch_rate   = Param(model.BRANCH, initialize={k: branch_data[k]['rate'] for k in branch_data if branch_data[k]['rate'] is not None})

model.G     = Var(model.GEN)
model.pk    = Var(model.BRANCH)
model.theta = Var(model.BUS)

# Add Suffix to retrieve dual variables (shadow prices = LMPs)
model.dual  = Suffix(direction=Suffix.IMPORT)

def obj_rule(m):
    return sum(m.gen_OpCost[g] * m.G[g] for g in m.GEN)
model.obj = Objective(rule=obj_rule, sense=minimize)

def branchLimits_rule(m, k):
    if branch_data[k]['rate'] is None: return Constraint.Skip
    return inequality(-m.branch_rate[k], m.pk[k], m.branch_rate[k])
model.branchLimits = Constraint(model.BRANCH, rule=branchLimits_rule)

def lineFlowEqs_rule(m, k):
    return m.pk[k] / BaseMW == (
        m.theta[m.branch_frmbus[k]] - m.theta[m.branch_tobus[k]]
    ) / m.branch_x[k]
model.lineFlowEqs = Constraint(model.BRANCH, rule=lineFlowEqs_rule)

def genLimits_rule(m, g):
    return inequality(m.gen_min[g], m.G[g], m.gen_max[g])
model.genLimits = Constraint(model.GEN, rule=genLimits_rule)

def NodalPowerBalance_rule(m, n):
    gen_p  = sum(m.G[g]  for g in m.GEN    if m.gen_bus[g]       == n)
    flow_in  = sum(m.pk[k] for k in m.BRANCH if m.branch_tobus[k]  == n)
    flow_out = sum(m.pk[k] for k in m.BRANCH if m.branch_frmbus[k] == n)
    return gen_p + flow_in - flow_out == m.busLoad[n]
model.NodalPowerBalance = Constraint(model.BUS, rule=NodalPowerBalance_rule)

model.theta[refBus].fix(0)


In [ ]:
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

start_time = time.time()
results = solver.solve(model, tee=True)
solve_time = time.time() - start_time

print('\n=== Dispatch ===')
for g in model.GEN:
    print(f'  G[{g}] (Bus {gen_data[g]["bus"]}) = {model.G[g].value:.4f} MW')

print('\n=== Branch Flows ===')
for k in model.BRANCH:
    print(f'  pk[{k}] (Bus {branch_data[k]["from"]}→{branch_data[k]["to"]}) = {model.pk[k].value:.4f} MW')

print('\n=== Nodal LMPs ($/MWh) ===')
for n in model.BUS:
    lmp = model.dual[model.NodalPowerBalance[n]]
    print(f'  LMP[Bus {n}] = {lmp:.4f}')

total_cost = sum(gen_data[g]['c'] * model.G[g].value for g in model.GEN)
print(f'\n=== Total Cost = ${total_cost:.2f} ===')
print(f'Total solve elapsed time: {solve_time:.4f} seconds')
